In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Git hub is the bridge connecting PyTorch and Colab

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/facial-visual-profile")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/DeweyYip/facial-visual-profile.git",
            str(repo),
        ],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        check=True,
    )

os.chdir(repo)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

print("Working directory:", Path.cwd())
print("Model file exists:", Path("models/resnet18.py").is_file())

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

source = Path("/content/drive/MyDrive/FacialVisualProfile/data")
required = [
    "img_align_celeba.zip",
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
]

for name in required:
    path = source / name
    if path.is_file():
        print(f"FOUND: {name} ({path.stat().st_size / 1024**2:.1f} MiB)")
    else:
        print(f"MISSING: {name}")

Unzip the images

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo = Path("/content/facial-visual-profile")
source = Path("/content/drive/MyDrive/FacialVisualProfile/data")
destination = repo / "data/raw"

assert repo.is_dir(), "Repository not found. Run the clone cell first."
destination.mkdir(parents=True, exist_ok=True)

required = [
    "img_align_celeba.zip",
    "list_attr_celeba.txt",
    "list_landmarks_align_celeba.txt",
    "list_eval_partition.txt",
]

for name in required:
    src = source / name
    dst = destination / name
    print(f"Copying: {name}", flush=True)
    shutil.copyfile(src, dst)

print("Extracting images...", flush=True)
subprocess.run(
    [
        "unzip", "-oq",
        str(destination / "img_align_celeba.zip"),
        "-d", str(destination),
    ],
    check=True,
)

image_dir = destination / "img_align_celeba"
count = sum(1 for p in image_dir.glob("*.jpg") if p.is_file())
print("Extracted images:", count)
assert count == 202599, f"Unexpected image count: {count}"

os.chdir(repo)
print("Working directory:", Path.cwd())
print("Local data preparation passed.")

At the current stage:
We have raw data in both places

FacialVisualProfile/data/	in Google Drive has the three images
/content/facial-visual-profile/data/raw/ inColab has ZIP、3 txt and the unzipped images used to train now

The following small test tests on one simple forward


In [ ]:
import os
import sys
import torch
from torch import nn
from torch.utils.data import DataLoader

repo = "/content/facial-visual-profile"
os.chdir(repo)
if repo not in sys.path:
    sys.path.insert(0, repo)

from datasets.celeba import CelebAAttributes
from models.resnet18 import ResNet18Attributes

assert torch.cuda.is_available(), "GPU is unavailable."
torch.manual_seed(42)
device = torch.device("cuda")

dataset = CelebAAttributes("train")
loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

images, targets, points, names = next(iter(loader))
images = images.to(device)
targets = targets.to(device)

model = ResNet18Attributes(
    num_attributes=len(dataset.attribute_names),
    pretrained=True,
).to(device)
model.eval()

with torch.inference_mode():
    logits = model(images)
    loss = nn.BCEWithLogitsLoss()(logits, targets)

assert logits.shape == targets.shape == (8, 24)
assert torch.isfinite(logits).all()
assert torch.isfinite(loss)

print("GPU:", torch.cuda.get_device_name(0))
print("Training samples:", len(dataset))
print("Images:", tuple(images.shape), images.device)
print("Targets:", tuple(targets.shape), targets.device)
print("Logits:", tuple(logits.shape), logits.device)
print("Initial BCE loss:", round(loss.item(), 6))
print("GPU forward check passed.")

Let's try one epoch

In [ ]:
import json
import platform
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from datasets.celeba import CelebAAttributes
from models.resnet18 import ResNet18Attributes

assert torch.cuda.is_available(), "GPU is unavailable."

drive_root = Path("/content/drive/MyDrive/FacialVisualProfile")
assert drive_root.is_dir(), "Mount Google Drive before running."

seed = 42
batch_size = 64  # 2544 batches in total
learning_rate = 0.0001

torch.manual_seed(seed)
device = torch.device("cuda")

train_dataset = CelebAAttributes("train")
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(seed),
    num_workers=2,
    pin_memory=True,
    drop_last=False,
)

model = ResNet18Attributes(
    num_attributes=len(train_dataset.attribute_names),
    pretrained=True,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.BCEWithLogitsLoss()

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir = drive_root / "runs" / f"day4_r18_{run_id}"
run_dir.mkdir(parents=True, exist_ok=False)

# training mode on
model.train()
total_loss = 0.0
samples_seen = 0

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()

progress = tqdm(train_loader, desc="Training epoch 1/1")

"""
There are 64 images for each batch
images: a [64, 3, 224, 224] tensor
targets: a [64, 24] tensor for the 24 attributes
points: for landmarks, not used in this test
names: the name for each image, not used in this test
enumerate makes it able to loop thru all the batches and tract the step number
"""
for step, (images, targets, points, names) in enumerate(progress, start=1):

    # Put the data onto the device 
    images = images.to(device, non_blocking=True)
    targets = targets.to(device, non_blocking=True)

    # clean up the gradient used in last step
    optimizer.zero_grad(set_to_none=True)
    # Getting the result
    logits = model(images)
    # Getting the loss
    loss = criterion(logits, targets)

    # check in case loss explodes
    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}")
    
    # backprop
    loss.backward()
    # gradients + Adam => new weights
    optimizer.step()

    count = images.size(0)                # answers how many images are there in a batch
    total_loss += loss.item() * count     # loss.item() is only the avg of one batch
    samples_seen += count                 # how many inmages the model has seen

    if step % 50 == 0 or step == len(train_loader):
        progress.set_postfix(
            mean_loss=f"{total_loss / samples_seen:.4f}"
        )

torch.cuda.synchronize() # need to be used before estimating time
elapsed = time.perf_counter() - started
peak_memory = torch.cuda.max_memory_allocated() / 1024**3

assert samples_seen == len(train_dataset)

report = {
    "purpose": "One-epoch training benchmark; no validation evaluation",
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "seed": seed,
    "weights": "ResNet18_Weights.IMAGENET1K_V1",
    "precision": "FP32",
    "optimizer": "Adam",
    "learning_rate": learning_rate,
    "batch_size": batch_size,
    "num_workers": 2,
    "epochs_completed": 1,
    "samples_seen": samples_seen,
    "mean_train_loss": total_loss / samples_seen,
    "epoch_seconds": elapsed,
    "images_per_second": samples_seen / elapsed,
    "peak_allocated_gpu_gib": peak_memory,
    "attribute_names": train_dataset.attribute_names,
    "preprocessing_source": Path(
        "datasets/preprocessing.py"
    ).read_text(encoding="utf-8"),
}

torch.save(
    {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "epoch": 1,
        "metadata": report,
    },
    run_dir / "checkpoint_epoch1.pt",
)

(run_dir / "benchmark.json").write_text(
    json.dumps(report, indent=2) + "\n", encoding="utf-8"
)

print(f"Samples seen: {samples_seen}")
print(f"Mean training loss: {report['mean_train_loss']:.6f}")
print(f"Epoch time: {elapsed / 60:.2f} minutes")
print(f"Throughput: {report['images_per_second']:.1f} images/second")
print(f"Peak allocated GPU memory: {peak_memory:.2f} GiB")
print("Saved checkpoint and report to:", run_dir)

In [ ]:
import json
from pathlib import Path

run_dir = Path(
    "/content/drive/MyDrive/FacialVisualProfile/runs/"
    "day4_r18_20260927T115428425737Z"
)

for name in ("checkpoint_epoch1.pt", "benchmark.json"):
    path = run_dir / name
    assert path.is_file(), f"Missing file: {path}"
    assert path.stat().st_size > 0, f"Empty file: {path}"
    print(f"{name}: {path.stat().st_size / 1024**2:.2f} MiB")

report = json.loads(
    (run_dir / "benchmark.json").read_text(encoding="utf-8")
)
assert report["samples_seen"] == 162770
assert report["epochs_completed"] == 1
print("Saved report check passed.")